In [3]:
import numpy as np

# Google Drive Setup

In [4]:
from google.colab import drive
import os

# Mount the drive to Colab
drive.mount('/content/drive')

# Get the path github private token
token_path = '/content/drive/MyDrive/.gh_token'

# Load the token
with open(token_path) as f:
    os.environ['GH_TOKEN'] = f.read().strip()

Mounted at /content/drive


In [5]:
print('GH_TOKEN' in os.environ)

True


# Github Setup

In [6]:
!git config --global user.email "kyle990987@gmail.com"
!git config --global user.name "shigenogoro"

In [7]:

# Set repo URL
username = "shigenogoro"
reponame = "CS_646_final_project"
token = os.environ['GH_TOKEN']

repo_url = f"https://{token}@github.com/{username}/{reponame}.git"

!git clone {repo_url}
%cd {reponame}

Cloning into 'CS_646_final_project'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 8 (delta 0), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 407.97 KiB | 40.80 MiB/s, done.
/content/CS_646_final_project


# Minimal Plan

1. Download a QA dataset subset (e.g., 1k HotpotQA queries).

2. Build a retriever (either DPR + FAISS or BM25) and index passages.

3. For each query: retrieve top-k passages, concatenate them, feed them to a generator model (RAG / T5 / Flan-T5).

4. Compute EM / F1 / ROUGE-L on outputs.
This matches the “Standard RAG” baseline from your proposal.

# Baseline

1. Standard RAG (Vanilla RAG without external Judge)

    The standard RAG relies on the retriever to retrieve the top-k documents that are fed directly into the generator, without any filtering, validation, or judging mechanisms. This is the simplest RAG setup serving as a lower bound in terms of accuracy and demonstrating values and costs added by any Judge or filtering mechanisms.

    Procedure:

    Query → Retriever → Top-k Documents → Generator → Answer

2. Standard RAG integrated with retriever-filtering (Standard RAG with score cutoff, without external Judge)

    Here, we apply a similarity-based score threshold to the top-k documents retrieved from the retriever, only passing those with a score higher than the threshold as input to the generator. This method utilizes the built-in scoring mechanism of the retriever to filter out documents with lower relevance without using an external Judge [ 8]. This baseline helps establish a sense of whether a basic filtering feature alone is sufficient for mitigating retrieval noise and a good balance between costs and performance.

    Procedure:

    Query → Retriever → Score Cutoff → Generator → Answer

3. MAIN-RAG (Full multi-agent RAG with LLM Judge and adaptive thresholds)

    The MAIN-RAG is a more recent training-free multi-agent RAG framework for selecting relevant documents incorporating a predictor, an LLM-based Judge, and adaptive filtering thresholds. We adopt MAIN-RAG as a strong baseline serving as an upper bound of accuracy and costs and a gold standard for evaluating trade-offs of using alternative Judge implementations.

## 1. Standard RAG (Vanilla RAG without external Judge)

### Package Setup

- `sentence-transformers`: For dense retriever

- `transformers`: Hugging Face Model Framework

- `datasets`: For Loading QA datasets

- `faiss-cpu`: For effective similarity search

- `accelerate`: For GPU support

- `rank-bm25`: For BM25 Retriever

In [8]:
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U datasets
!pip install -U faiss-cpu
!pip install rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 62.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 88.4 MB/s eta 0:

### Load Datasets

1. HotpotQA

2. TriviaQA

3. Natural Questions (NQ)

4. BioASQ

In [9]:
# Load datasets
from datasets import load_dataset

# Load HotpotQA
hotpotqa = load_dataset("hotpot_qa", "fullwiki", split="validation[:100]")
print("Load HotpotQA Successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Load HotpotQA Successfully.


In [10]:
# Load TriviaQA
triviaqa = load_dataset("trivia_qa", "rc", split="validation[:100]")
print("Load TriviaQA Successfully.")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc/train-00000-of-00026.parquet:   0%|          | 0.00/308M [00:00<?, ?B/s]

rc/train-00001-of-00026.parquet:   0%|          | 0.00/298M [00:00<?, ?B/s]

rc/train-00002-of-00026.parquet:   0%|          | 0.00/290M [00:00<?, ?B/s]

... (31 outputs left out)

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Load TriviaQA Successfully.


In [ ]:
# Load Natural Questions
natural_questions = load_dataset("natural_questions", split="validation[:100]")
print("Load Natural Questions Successfully.")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

default/train-00000-of-00287.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

default/train-00001-of-00287.parquet:   0%|          | 0.00/202M [00:00<?, ?B/s]

default/train-00002-of-00287.parquet:   0%|          | 0.00/197M [00:00<?, ?B/s]

... (93 outputs left out)

default/train-00096-of-00287.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

default/train-00097-of-00287.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

default/train-00098-of-00287.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

default/train-00099-of-00287.parquet:   0%|          | 0.00/197M [00:00<?, ?B/s]

In [ ]:
# Load BioASQ
bioasq = load_dataset("bioasq", "task_b", split="train[:100]")
print("Load BioASQ Successfully.")

### Retriever

1. BM25

2. DPR (Dense Passage Retriever)

3. Contriever (Self-supervised Dense Retriever)

In [ ]:
# Create a BM25 Retriever
from rank_bm25 import BM25Okapi

def bm25_retrieve(query, corpus, top_k=3):
    tokenized_corpus = [doc.split(" ") for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = query.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_docs_indices = np.argsort(doc_scores)[::-1][:top_k]
    top_docs = [corpus[i] for i in top_docs_indices]
    return top_docs

In [ ]:
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoder, DPRContextEncoderTokenizer
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load two encoders
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base").to("cuda")

c_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
c_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base").to("cuda")

def dpr_retrieve(query, corpus, top_k=3):
    # Encode query
    q_inputs = q_tokenizer(query, return_tensors="pt").to("cuda")
    q_emb = q_encoder(**q_inputs).pooler_output.cpu().detach().numpy()

    # Encode corpus
    c_inputs = c_tokenizer(corpus, return_tensors="pt", padding=True, truncation=True).to("cuda")
    c_embs = c_encoder(**c_inputs).pooler_output.cpu().detach().numpy()

    # Compute similarity
    sims = cosine_similarity(q_emb, c_embs)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    top_docs = [corpus[i] for i in top_idx]
    return top_docs

In [ ]:
contriever_model = SentenceTransformer("facebook/contriever")

def contriever_retrieve(query, corpus, top_k=3):
    corpus_emb = contriever_model.encode(corpus, convert_to_tensor=True)
    query_emb = contriever_model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, corpus_emb, top_k=top_k)[0]
    top_docs = [corpus[hit['corpus_id']] for hit in hits]
    return top_docs

### Generator

- Use FLAN-T5

In [ ]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base", device=0)

### Integrate into RAG Pipeline

In [ ]:
def rag_answer(query, corpus, retriever_fn, generator, top_k=3):
    docs = retriever_fn(query, corpus, top_k=top_k)
    context = " ".join(docs)
    prompt = f"Question: {query}\nContext: {context}\nAnswer:"
    answer = generator(prompt, max_new_tokens=128, do_sample=False)[0]['generated_text']
    return answer

### Run testing on each dataset using different retriever

In [2]:
# Run retrievers on HotpotQA

sample = hotpotqa[0]
question = sample["question"]
# Ensure corpus is a list of strings, where each string is a document
corpus = [" ".join(sentences) for sentences in sample["context"]["sentences"]]
gold = sample["answer"]

print("Question:", question)
print("Gold Answer:", gold)

# for name, retriever in [("BM25", bm25_retrieve), ("DPR", dpr_retrieve), ("Contriever", contriever_retrieve)]:
for name, retriever in [("BM25", bm25_retrieve), ("Contriever", contriever_retrieve)]:
    # For BM25, the corpus is already in the correct format (list of strings)
    current_corpus = corpus
    answer = rag_answer(question, current_corpus, retriever, generator)
    print(f"\n[{name}] → {answer}")

NameError: name 'hotpotqa' is not defined

### Evaluation

1. QA Metrics

2. Filtering Metrics

## 2. Standard RAG integrated with retriever-filtering (Standard RAG with score cutoff, without external Judge)

## 3. MAIN-RAG (Full multi-agent RAG with LLM Judge and adaptive thresholds)

# Commit Changes

In [ ]:
# Commit + push
!git add -A

In [ ]:
!git commit -m "Update Baseline Notebook with GitHub Settings and TODO"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
!git push {repo_url} main